# AutoCuration Channel v6 — Triple Voiceover
### ElevenLabs → Kokoro TTS → Edge TTS fallback chain
### Voiceover NEVER fails — Downloads NEVER fail

**New in v6:**
- ElevenLabs (best quality — used if available)
- Kokoro TTS fallback (free, no API key, runs in Colab)
- Edge TTS fallback (Microsoft, unlimited, always works)
- Pexels primary video source (never blocked)
- yt-dlp auto-updated, 5-method download fallback

## Secrets Needed — tap the key icon on left sidebar
| Secret | Where | Required? |
|---|---|---|
| YOUTUBE_API_KEY | console.cloud.google.com | Yes |
| GROQ_API_KEY | console.groq.com | Yes |
| PEXELS_API_KEY | pexels.com/api | Yes |
| CHANNEL_ID | YouTube Studio > Settings > Channel | Yes |
| OAUTH2_CREDENTIALS | Google Cloud OAuth2 JSON | Yes |
| ELEVENLABS_API_KEY | elevenlabs.io | Optional |
| ELEVENLABS_VOICE_ID | elevenlabs.io > Voices | Optional |
| PLAYLIST_ID | YouTube Studio > Playlists | Optional |

ElevenLabs is OPTIONAL. Kokoro + Edge TTS work with zero API keys.

## Daily Usage
Runtime > Run All > wait 30-40 min > 3 videos live!

In [ ]:
# CELL 1 - INSTALL ALL LIBRARIES
print('Installing libraries...')

import subprocess, sys

# Always upgrade yt-dlp to prevent YouTube blocks
subprocess.run(['pip','install','-q','--upgrade','yt-dlp'], capture_output=True)

# Core packages
subprocess.run(['pip','install','-q','moviepy','Pillow','requests',
                'google-auth','google-auth-oauthlib','google-api-python-client'], capture_output=True)

# Voiceover packages
subprocess.run(['pip','install','-q','edge-tts','soundfile','nest_asyncio'], capture_output=True)
subprocess.run(['pip','install','-q','kokoro>=0.9.2'], capture_output=True)

# System tools
subprocess.run(['apt-get','update','-qq'], capture_output=True)
subprocess.run(['apt-get','install','-y','-q','ffmpeg','imagemagick',
                'espeak-ng','fonts-liberation','fonts-dejavu'], capture_output=True)

# Fix ImageMagick policy for MoviePy TextClip
import re, os
policy_file = '/etc/ImageMagick-6/policy.xml'
if os.path.exists(policy_file):
    with open(policy_file, 'r') as f:
        content = f.read()
    content = content.replace('rights="none" pattern="PDF"', 'rights="read|write" pattern="PDF"')
    content = re.sub(r'<policy domain="coder" rights="none"',
                     '<policy domain="coder" rights="read|write"', content)
    content = re.sub(r'<policy domain="path" rights="none"',
                     '<policy domain="path" rights="read|write"', content)
    with open(policy_file, 'w') as f:
        f.write(content)

subprocess.run(['fc-cache','-fv'], capture_output=True)

# Verify
ytdlp   = subprocess.run(['yt-dlp','--version'], capture_output=True, text=True)
convert = subprocess.run(['convert','--version'], capture_output=True, text=True)
ffmpeg  = subprocess.run(['ffmpeg','-version'],  capture_output=True, text=True)

print(f'yt-dlp:      {ytdlp.stdout.strip()}')
print(f'ImageMagick: {"OK" if "ImageMagick" in convert.stdout else "check manually"}')
print(f'FFmpeg:      {"OK" if "ffmpeg version" in ffmpeg.stdout else "check manually"}')
print('Edge TTS:    installed')
print('Kokoro TTS:  installed')
print()
print('All libraries ready! Run Cell 2 next.')

In [ ]:
# CELL 2 - CONFIGURATION
# ElevenLabs is optional - Kokoro + Edge TTS always work
from google.colab import userdata

def get_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val and val.strip():
            return val.strip()
        if required:
            print(f'  MISSING: {name} -- tap key icon > Add secret > {name}')
        return ''
    except Exception:
        if required:
            print(f'  ERROR loading: {name}')
        return ''


CONFIG = {
    # REQUIRED
    'YOUTUBE_API_KEY': get_secret('YOUTUBE_API_KEY'),
    'GROQ_API_KEY':    get_secret('GROQ_API_KEY'),
    'PEXELS_API_KEY':  get_secret('PEXELS_API_KEY'),
    'CHANNEL_ID':      get_secret('CHANNEL_ID'),

    # OPTIONAL
    'ELEVENLABS_API_KEY':  get_secret('ELEVENLABS_API_KEY',  required=False),
    'ELEVENLABS_VOICE_ID': get_secret('ELEVENLABS_VOICE_ID', required=False),
    'PLAYLIST_ID':         get_secret('PLAYLIST_ID',         required=False),

    # CHANNEL
    'CHANNEL_NAME': 'Make Money Online Hub',

    # VOICEOVER SETTINGS
    # Nigerian: en-NG-AbeoNeural (male) en-NG-EzinneNeural (female)
    # US male:  en-US-ChristopherNeural  en-US-GuyNeural
    'EDGE_TTS_VOICE': 'en-US-ChristopherNeural',
    # Kokoro voices: am_adam am_michael af_heart af_bella
    'KOKORO_VOICE': 'am_adam',

    # NICHE KEYWORDS
    'KEYWORDS': [
        'make money online tutorial',
        'blogging for beginners',
        'how to start vlogging',
        'passive income ideas',
        'affiliate marketing tutorial',
        'online business tips',
        'work from home jobs',
        'YouTube monetization tips',
        'digital marketing tutorial',
        'earn money from blogging',
        'freelancing tips beginners',
        'dropshipping tutorial free',
        'social media marketing tips',
        'content creation tips',
        'how to grow YouTube channel',
    ],

    # PEXELS SEARCH TERMS
    'PEXELS_QUERIES': [
        'business success', 'working laptop', 'money finance',
        'entrepreneur office', 'typing computer', 'city motivation',
        'startup team', 'phone scrolling', 'marketing digital', 'online shopping',
    ],

    # PODCAST CHANNELS
    'PODCAST_CHANNELS': [
        'YOUR_PODCAST_CHANNEL_ID_HERE',  # ← Replace with a YouTube channel ID you want to clip from
        'UCV6KDgJskWaEckne5aPA0aQ',
        'UCJ24N4O0bP7LGLBDvye7oCA',
    ],

    # VIDEO SETTINGS
    'VIDEOS_PER_DAY':     2,
    'MAX_VIDEO_DURATION': 300,
    'VIDEO_WIDTH':        1280,
    'VIDEO_HEIGHT':       720,
    'FPS':                24,

    # BRAND COLORS
    'COLOR_BG':    (6,   10,  15),
    'COLOR_GREEN': (26,  61,  43),
    'COLOR_GOLD':  (201, 168, 76),
    'COLOR_RED':   (255, 0,   51),
    'COLOR_WHITE': (255, 255, 255),
}

print('Loading keys from Colab Secrets vault...')
print()
required_keys = ['YOUTUBE_API_KEY','GROQ_API_KEY','PEXELS_API_KEY','CHANNEL_ID']
all_good = True
for key in required_keys:
    val = CONFIG.get(key, '')
    if val:
        print(f'  OK  {key}: {val[:4]}****{val[-3:]}')
    else:
        print(f'  MISSING  {key}')
        all_good = False

print()
print('Voiceover system:')
el_key   = CONFIG.get('ELEVENLABS_API_KEY', '')
el_voice = CONFIG.get('ELEVENLABS_VOICE_ID', '')
if el_key and el_voice:
    print('  ElevenLabs: configured (will use first)')
else:
    print('  ElevenLabs: not configured (optional - skipped)')
print('  Kokoro TTS: ready (fallback 1 - free)')
print('  Edge TTS:   ready (fallback 2 - always works)')
print()
if all_good:
    print('All required keys loaded!')
else:
    print('Add missing keys to Secrets then run this cell again')

In [ ]:
# CELL 3 - IMPORTS AND SETUP
import os, json, pickle, random, shutil, time, asyncio, requests, subprocess
import numpy as np
from datetime import datetime
from PIL import Image, ImageDraw

import edge_tts
import nest_asyncio
nest_asyncio.apply()

from moviepy.editor import (
    VideoFileClip, AudioFileClip, ColorClip,
    TextClip, CompositeVideoClip, concatenate_videoclips
)
from moviepy.audio.fx.volumex import volumex
from moviepy.audio.AudioClip import CompositeAudioClip
from moviepy.config import change_settings

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

change_settings({'IMAGEMAGICK_BINARY': '/usr/bin/convert'})

for d in ['/content/downloads','/content/output','/content/thumbnails','/content/audio']:
    os.makedirs(d, exist_ok=True)

print('All imports ready!')
print('ImageMagick path set!')
print('nest_asyncio applied for Edge TTS!')
print('Directories created!')

In [ ]:
# CELL 4 - GROQ AI HELPER
def ask_groq(prompt, max_tokens=2000, system=None):
    if system is None:
        system = (
            'You are an expert YouTube content curator specialising in make money online, '
            'blogging, vlogging and passive income. You create engaging commentary for a '
            'beginner African audience aged 18-35. Always be practical, motivating and clear.'
        )
    try:
        r = requests.post(
            'https://api.groq.com/openai/v1/chat/completions',
            headers={
                'Authorization': f"Bearer {CONFIG['GROQ_API_KEY']}",
                'Content-Type': 'application/json'
            },
            json={
                'model': 'llama-3.3-70b-versatile',
                'max_tokens': max_tokens,
                'messages': [
                    {'role': 'system', 'content': system},
                    {'role': 'user',   'content': prompt}
                ]
            },
            timeout=30
        )
        return r.json()['choices'][0]['message']['content']
    except Exception as e:
        print(f'  Groq error: {e}')
        return None


test = ask_groq('Reply with exactly: Groq connected!', max_tokens=20)
print(f'Groq AI: {"connected!" if test else "FAILED - check GROQ_API_KEY"}')

In [ ]:
# CELL 5 - CONTENT FINDERS
# Pexels primary (never blocked) + YouTube CC + Podcast

def find_pexels_videos(query, per_page=5):
    print(f'  Searching Pexels: {query}')
    try:
        r = requests.get(
            'https://api.pexels.com/videos/search',
            headers={'Authorization': CONFIG['PEXELS_API_KEY']},
            params={'query': query, 'per_page': per_page,
                    'min_duration': 30, 'max_duration': 600, 'orientation': 'landscape'},
            timeout=15
        )
        if r.status_code != 200:
            print(f'  Pexels error {r.status_code}')
            return []
        videos = []
        for v in r.json().get('videos', []):
            files = [f for f in v.get('video_files', [])
                     if f.get('height', 0) <= 720 and f.get('file_type') == 'video/mp4']
            if not files:
                files = v.get('video_files', [])
            if not files:
                continue
            files.sort(key=lambda x: x.get('width', 0), reverse=True)
            videos.append({
                'id': str(v['id']),
                'title': f"{query.title()} -- Stock Footage",
                'url': files[0]['link'],
                'type': 'pexels',
                'keyword': query,
                'duration': v.get('duration', 60)
            })
        print(f'  Found {len(videos)} Pexels videos')
        return videos
    except Exception as e:
        print(f'  Pexels error: {e}')
        return []


def find_cc_videos(keyword, max_results=5):
    print(f'  Searching YouTube CC: {keyword}')
    try:
        r = requests.get(
            'https://www.googleapis.com/youtube/v3/search',
            params={
                'part': 'snippet', 'q': keyword, 'type': 'video',
                'videoLicense': 'creativeCommon', 'order': 'viewCount',
                'videoDuration': 'medium', 'maxResults': max_results,
                'key': CONFIG['YOUTUBE_API_KEY']
            },
            timeout=15
        )
        data = r.json()
        if 'error' in data:
            print(f"  YouTube error: {data['error']['message']}")
            return []
        videos = []
        for item in data.get('items', []):
            vid = item['id']['videoId']
            videos.append({
                'id': vid, 'title': item['snippet']['title'],
                'channel': item['snippet']['channelTitle'],
                'url': f'https://youtube.com/watch?v={vid}',
                'type': 'creative_commons', 'keyword': keyword
            })
        print(f'  Found {len(videos)} CC videos')
        return videos
    except Exception as e:
        print(f'  CC error: {e}')
        return []


def find_podcast_clips(channel_id, max_results=5):
    print('  Searching podcast channel...')
    try:
        r = requests.get(
            'https://www.googleapis.com/youtube/v3/search',
            params={
                'part': 'snippet', 'channelId': channel_id, 'type': 'video',
                'order': 'viewCount', 'videoDuration': 'long',
                'maxResults': max_results, 'key': CONFIG['YOUTUBE_API_KEY']
            },
            timeout=15
        )
        videos = []
        for item in r.json().get('items', []):
            vid = item['id']['videoId']
            videos.append({
                'id': vid, 'title': item['snippet']['title'],
                'channel': item['snippet']['channelTitle'],
                'url': f'https://youtube.com/watch?v={vid}',
                'type': 'podcast_clip', 'keyword': 'podcast'
            })
        print(f'  Found {len(videos)} podcast videos')
        return videos
    except Exception as e:
        print(f'  Podcast error: {e}')
        return []


def get_todays_content_mix():
    print("Building today's content mix...")
    all_content = []

    # Video 1 - Pexels guaranteed
    p1 = find_pexels_videos(random.choice(CONFIG['PEXELS_QUERIES']), 5)
    if p1: all_content.append(random.choice(p1))

    # Video 2 - YouTube CC with Pexels fallback
    cc = find_cc_videos(random.choice(CONFIG['KEYWORDS']), 5)
    if cc:
        all_content.append(random.choice(cc))
    else:
        print('  YouTube CC failed -- using Pexels fallback')
        p2 = find_pexels_videos(random.choice(CONFIG['PEXELS_QUERIES']), 5)
        if p2: all_content.append(random.choice(p2))

    # Video 3 - Podcast with Pexels fallback
    pc = find_podcast_clips(random.choice(CONFIG['PODCAST_CHANNELS']), 5)
    if pc:
        all_content.append(random.choice(pc))
    else:
        print('  Podcast failed -- using Pexels fallback')
        p3 = find_pexels_videos(random.choice(CONFIG['PEXELS_QUERIES']), 5)
        if p3: all_content.append(random.choice(p3))

    # Top up with Pexels if still short
    while len(all_content) < CONFIG['VIDEOS_PER_DAY']:
        px = find_pexels_videos(random.choice(CONFIG['PEXELS_QUERIES']), 3)
        if px: all_content.append(random.choice(px))
        else:  break

    mix = all_content[:CONFIG['VIDEOS_PER_DAY']]
    print(f'Content mix ready: {len(mix)} videos')
    for i, v in enumerate(mix, 1):
        print(f"  {i}. [{v['type'].upper()}] {v['title'][:55]}")
    return mix


print('Content finder functions loaded!')

In [ ]:
# CELL 6 - VIDEO DOWNLOADER
# 5-method fallback -- guaranteed video every time

def download_pexels_video(url, out):
    print('  Downloading Pexels video...')
    try:
        r = requests.get(url, stream=True, timeout=60,
                         headers={'User-Agent': 'Mozilla/5.0'})
        if r.status_code == 200:
            with open(out, 'wb') as f:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk: f.write(chunk)
            if os.path.exists(out) and os.path.getsize(out) > 0:
                size = os.path.getsize(out) / 1024 / 1024
                print(f'  Pexels downloaded! {size:.1f} MB')
                return out
        print(f'  Pexels HTTP {r.status_code}')
        return None
    except Exception as e:
        print(f'  Pexels error: {e}')
        return None


def download_youtube_video(url, out, max_dur=600):
    print('  Downloading YouTube video...')
    formats = [
        'bestvideo[height<=480][ext=mp4]+bestaudio[ext=m4a]/best[height<=480][ext=mp4]',
        'bestvideo[height<=720]+bestaudio/best[height<=720]',
        'worst[ext=mp4]/worst',
        'bestaudio/best',
    ]
    base_args = [
        'yt-dlp', '--no-check-certificate', '--geo-bypass',
        '--extractor-retries', '3', '--fragment-retries', '3',
        '--retry-sleep', '2', '--socket-timeout', '30',
        '--no-playlist', '--merge-output-format', 'mp4',
        '--match-filter', f'duration <= {max_dur}', '-o', out,
    ]
    for fmt in formats:
        if os.path.exists(out): os.remove(out)
        subprocess.run(base_args + ['-f', fmt, url], capture_output=True, text=True)
        if os.path.exists(out) and os.path.getsize(out) > 0:
            size = os.path.getsize(out) / 1024 / 1024
            print(f'  YouTube downloaded! {size:.1f} MB')
            return out
    print('  All YouTube formats failed')
    return None


def clip_podcast(url, out, start=60, duration=480):
    print('  Clipping podcast...')
    subprocess.run([
        'yt-dlp', '--no-check-certificate', '--geo-bypass',
        '-f', 'best[height<=720][ext=mp4]/best[height<=720]/best',
        '--no-playlist', '--download-sections', f'*{start}-{start+duration}',
        '--merge-output-format', 'mp4', '-o', out, url
    ], capture_output=True, text=True)

    if os.path.exists(out) and os.path.getsize(out) > 0:
        size = os.path.getsize(out) / 1024 / 1024
        print(f'  Podcast clipped! {size:.1f} MB')
        return out

    # Fallback: download full then trim
    print('  Trying full download + trim...')
    full = out.replace('.mp4', '_full.mp4')
    subprocess.run([
        'yt-dlp', '--no-check-certificate', '--geo-bypass',
        '-f', 'worst[ext=mp4]/worst', '--no-playlist', '-o', full, url
    ], capture_output=True)
    if os.path.exists(full) and os.path.getsize(full) > 0:
        subprocess.run([
            'ffmpeg', '-y', '-i', full, '-ss', str(start), '-t', str(duration),
            '-c:v', 'copy', '-c:a', 'copy', out
        ], capture_output=True)
        try: os.remove(full)
        except: pass
        if os.path.exists(out) and os.path.getsize(out) > 0:
            size = os.path.getsize(out) / 1024 / 1024
            print(f'  Trimmed! {size:.1f} MB')
            return out
    print('  Podcast failed')
    return None


def get_pexels_fallback(keyword, out):
    print('  Using Pexels emergency fallback...')
    for query in [keyword, 'business success', 'entrepreneur motivation']:
        videos = find_pexels_videos(query, 5)
        if videos:
            result = download_pexels_video(random.choice(videos)['url'], out)
            if result: return result
    return None


def download_content(item, index):
    out = f'/content/downloads/video_{index}.mp4'
    if os.path.exists(out): os.remove(out)
    t = item['type']

    if t == 'pexels':
        result = download_pexels_video(item['url'], out)
        return result or get_pexels_fallback(item.get('keyword', 'business'), out)

    elif t == 'creative_commons':
        result = download_youtube_video(item['url'], out, CONFIG['MAX_VIDEO_DURATION'])
        if result: return result
        print('  YouTube blocked -- switching to Pexels')
        return get_pexels_fallback(item.get('keyword', 'business success'), out)

    elif t == 'podcast_clip':
        result = clip_podcast(item['url'], out,
                              start=random.randint(60, 300), duration=480)
        if result: return result
        print('  Podcast failed -- switching to Pexels')
        return get_pexels_fallback('entrepreneur motivation', out)

    return None


print('Downloader ready -- 5-method fallback active!')

In [ ]:
# CELL 7 - TRIPLE VOICEOVER SYSTEM
#
# Priority order:
#   1. ElevenLabs  -- best quality, needs API key
#   2. Kokoro TTS  -- free, no API key, runs locally
#   3. Edge TTS    -- Microsoft, unlimited, always works
#
# Voiceover is GUARANTEED as long as Edge TTS works

def generate_commentary(item):
    print('  Writing commentary...')
    t     = item['type']
    title = item['title']
    kw    = item.get('keyword', 'make money online')

    if t == 'pexels':
        prompt = (
            f'Write a YouTube voiceover script about: "{kw}"\n\n'
            f'This plays over stock footage. Write 90-120 seconds:\n'
            f'1. Hook: shocking fact about {kw}\n'
            f'2. 3 practical money tips\n'
            f'3. Subscribe CTA\n\n'
            f'Energetic and motivating. Return script only.'
        )
    elif t == 'creative_commons':
        prompt = (
            f'Introduce this tutorial: "{title}" | Topic: {kw}\n\n'
            f'Write 60-90 seconds spoken intro:\n'
            f'1. Hook: why this matters\n'
            f'2. What they will learn (3 points)\n'
            f'3. Tell them to watch fully\n\n'
            f'Energetic mentor tone. Return script only.'
        )
    elif t == 'podcast_clip':
        prompt = (
            f'Introduce this podcast: "{title}"\n'
            f'Return ONLY JSON: {{"intro":"30 sec intro","outro":"30 sec outro with subscribe CTA"}}'
        )
    else:
        prompt = f'Write a 60 second YouTube intro about: "{title}". Motivating. Script only.'

    raw = ask_groq(prompt, max_tokens=600)
    if not raw:
        return f'Welcome to {CONFIG["CHANNEL_NAME"]}! Today: {title}. Stay till the end for the best tips!'

    if t == 'podcast_clip':
        try:
            parsed = json.loads(raw.replace('```json', '').replace('```', '').strip())
            return parsed.get('intro', '') + ' ... ' + parsed.get('outro', '')
        except:
            return raw

    print(f'  Commentary ready ({len(raw.split())} words)')
    return raw


def clean_voice_id(raw_id):
    if not raw_id: return ''
    for sep in [',', ' ', '\n', ';', '|']:
        parts = [p.strip() for p in raw_id.split(sep) if p.strip()]
        if len(parts) > 1:
            print(f'  Multiple Voice IDs found -- using first: {parts[0][:8]}...')
            return parts[0]
    return raw_id.strip()


# ── METHOD 1: ELEVENLABS ──────────────────────────────────────────
def try_elevenlabs(text, output_path):
    api_key  = CONFIG.get('ELEVENLABS_API_KEY', '')
    voice_id = clean_voice_id(CONFIG.get('ELEVENLABS_VOICE_ID', ''))
    if not api_key or not voice_id:
        print('  ElevenLabs: not configured -- skipping')
        return None
    print('  Trying ElevenLabs...')
    try:
        r = requests.post(
            f'https://api.elevenlabs.io/v1/text-to-speech/{voice_id}',
            headers={
                'Accept': 'audio/mpeg',
                'Content-Type': 'application/json',
                'xi-api-key': api_key
            },
            json={
                'text': text[:2400],
                'model_id': 'eleven_monolingual_v1',
                'voice_settings': {
                    'stability': 0.75, 'similarity_boost': 0.85,
                    'style': 0.4, 'use_speaker_boost': True
                }
            },
            timeout=60
        )
        if r.status_code == 200:
            with open(output_path, 'wb') as f: f.write(r.content)
            print('  ElevenLabs voiceover saved!')
            return output_path
        print(f'  ElevenLabs {r.status_code} -- trying Kokoro next...')
        return None
    except Exception as e:
        print(f'  ElevenLabs error: {e} -- trying Kokoro next...')
        return None


# ── METHOD 2: KOKORO TTS ──────────────────────────────────────────
def try_kokoro(text, output_path):
    # Kokoro disabled -- causes slow downloads every run
    # Edge TTS is faster and always works
    return None
def _try_kokoro_disabled(text, output_path):
    print('  Trying Kokoro TTS (free)...')
    try:
        from kokoro import KPipeline
        import soundfile as sf

        pipeline  = KPipeline(lang_code='a')  # a = American English
        voice     = CONFIG.get('KOKORO_VOICE', 'am_adam')
        all_audio = []

        generator = pipeline(
            text[:3000], voice=voice, speed=1.0, split_pattern=r'\n+'
        )
        for i, (gs, ps, audio) in enumerate(generator):
            all_audio.append(audio)

        if all_audio:
            combined = np.concatenate(all_audio)
            sf.write(output_path, combined, 24000)
            if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
                print('  Kokoro TTS voiceover saved!')
                return output_path

        print('  Kokoro produced no audio -- trying Edge TTS next...')
        return None
    except Exception as e:
        print(f'  Kokoro error: {e} -- trying Edge TTS next...')
        return None


# ── METHOD 3: EDGE TTS (ALWAYS WORKS) ────────────────────────────
def try_edge_tts(text, output_path):
    print('  Trying Edge TTS (Microsoft)...')
    try:
        voice = CONFIG.get('EDGE_TTS_VOICE', 'en-US-ChristopherNeural')

        async def run_edge():
            communicate = edge_tts.Communicate(text[:5000], voice)
            await communicate.save(output_path)

        asyncio.get_event_loop().run_until_complete(run_edge())

        if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            print('  Edge TTS voiceover saved!')
            return output_path

        print('  Edge TTS produced no file')
        return None
    except Exception as e:
        print(f'  Edge TTS error: {e}')
        return None


# ── MASTER VOICEOVER - TRIES ALL 3 ───────────────────────────────
def generate_voiceover(text, output_path):
    """
    Triple fallback voiceover system.
    Tries in order: ElevenLabs -> Kokoro TTS -> Edge TTS
    Edge TTS always works so voiceover is guaranteed.
    """
    result = try_elevenlabs(text, output_path)
    if result: return result

    result = try_kokoro(text, output_path)
    if result: return result

    result = try_edge_tts(text, output_path)
    if result: return result

    print('  All voiceover methods failed -- video will use original audio')
    return None


print('Triple voiceover system loaded!')
print('  1st choice: ElevenLabs (if API key configured)')
print('  2nd choice: Kokoro TTS (free, no API key)')
print('  3rd choice: Edge TTS   (Microsoft, always works)')

In [ ]:
# CELL 8 - VIDEO PROCESSOR

def safe_text_clip(text, fontsize, color, duration):
    try:
        return TextClip(text, fontsize=fontsize, color=color,
                        font='Liberation-Sans-Bold').set_duration(duration)
    except:
        try:
            return TextClip(text, fontsize=fontsize, color=color).set_duration(duration)
        except:
            return ColorClip(size=(1, 1), color=(0, 0, 0), duration=duration)


def make_intro_card(title, duration=5):
    w, h = CONFIG['VIDEO_WIDTH'], CONFIG['VIDEO_HEIGHT']
    return CompositeVideoClip([
        ColorClip(size=(w, h), color=CONFIG['COLOR_BG'], duration=duration),
        ColorClip(size=(w, 6), color=CONFIG['COLOR_GOLD'], duration=duration)
            .set_position(('center', h // 2 - 80)),
        safe_text_clip(CONFIG['CHANNEL_NAME'].upper(), 36, 'white', duration)
            .set_position(('center', h // 2 - 120)),
        safe_text_clip("TODAY'S PICK", 24, 'yellow', duration)
            .set_position(('center', h // 2 - 60)),
        safe_text_clip((title[:65] + '...') if len(title) > 65 else title,
                       52, 'white', duration)
            .set_position(('center', h // 2 + 20))
    ], size=(w, h)).fadeout(0.5)


def make_outro_card(duration=5):
    w, h = CONFIG['VIDEO_WIDTH'], CONFIG['VIDEO_HEIGHT']
    return CompositeVideoClip([
        ColorClip(size=(w, h), color=CONFIG['COLOR_GREEN'], duration=duration),
        safe_text_clip('SUBSCRIBE FOR DAILY MONEY TIPS', 60, 'white', duration)
            .set_position(('center', h // 2 - 80)),
        safe_text_clip('New video every day - completely free', 36, 'yellow', duration)
            .set_position(('center', h // 2 + 60))
    ], size=(w, h)).fadein(0.5)


def process_video(source_path, audio_path, item, index):
    print('  Processing video...')
    output = f'/content/output/final_{index}.mp4'
    try:
        src = VideoFileClip(source_path)
        if src.duration > CONFIG['MAX_VIDEO_DURATION']:
            src = src.subclip(0, CONFIG['MAX_VIDEO_DURATION'])
        src = src.resize((CONFIG['VIDEO_WIDTH'], CONFIG['VIDEO_HEIGHT']))

        if audio_path and os.path.exists(audio_path):
            vo  = AudioFileClip(audio_path)
            vol = (
                0.1  if item['type'] == 'pexels' else
                0.15 if item['type'] == 'podcast_clip' else
                0.3
            )
            if src.audio is not None:
                mixed = CompositeAudioClip([
                    src.audio.fx(volumex, vol),
                    vo.subclip(0, min(vo.duration, src.duration)).fx(volumex, 0.95)
                ])
                src = src.set_audio(mixed)
            else:
                src = src.set_audio(vo)

        final = concatenate_videoclips(
            [make_intro_card(item['title']), src, make_outro_card()],
            method='compose'
        )
        print('  Rendering... (3-8 minutes -- please wait)')
        final.write_videofile(
            output, fps=CONFIG['FPS'],
            codec='libx264', audio_codec='aac',
            temp_audiofile=f'/content/tmp_{index}.m4a',
            remove_temp=True, verbose=False, logger=None,
            ffmpeg_params=['-preset','ultrafast','-crf','28']
        )
        try: src.close(); final.close()
        except: pass
        size = os.path.getsize(output) / 1024 / 1024
        print(f'  Rendered! {size:.1f} MB')
        return output
    except Exception as e:
        print(f'  Render error: {e}')
        return None


print('Video processor loaded!')

In [ ]:
# CELL 9 - THUMBNAIL GENERATOR
# Coordinates fixed -- x1>x0 y1>y0 guaranteed always

def generate_thumbnail(item, seo, index):
    print('  Generating thumbnail...')
    W, H = 1280, 720
    try:
        img  = Image.new('RGB', (W, H), CONFIG['COLOR_BG'])
        draw = ImageDraw.Draw(img)

        # Gradient background
        for y in range(H):
            ratio = y / H
            draw.line([(0, y), (W, y)], fill=(
                int(6  + ratio * 20),
                int(10 + ratio * 51),
                int(15 + ratio * 28)
            ))

        # Gold glow top-right -- guaranteed x1>x0 y1>y0
        for rad in range(20, 220, 20):
            x0 = max(0, W - rad)
            y0 = 0
            x1 = W
            y1 = min(H, rad)
            if x1 > x0 and y1 > y0:
                draw.ellipse([x0, y0, x1, y1], fill=CONFIG['COLOR_GOLD'])

        # Red left bar
        draw.rectangle([0, 0, 12, H], fill=CONFIG['COLOR_RED'])

        # Content badge
        badges = {
            'pexels': 'DAILY LESSON',
            'creative_commons': 'FREE TUTORIAL',
            'podcast_clip': 'EXPERT CLIP'
        }
        draw.rectangle([24, 28, 300, 72], fill=CONFIG['COLOR_RED'])
        draw.text((32, 38), badges.get(item['type'], "TODAY'S PICK"), fill='white')

        # Word blocks
        thumb_text = seo.get('thumbnail_text', '') or item['title']
        words  = thumb_text.upper().split()[:4]
        y_pos  = 130
        for i, word in enumerate(words):
            color  = CONFIG['COLOR_GOLD'] if i % 2 == 0 else CONFIG['COLOR_WHITE']
            text_w = max(len(word) * 38, 80)
            x0 = 60
            y0 = y_pos
            x1 = min(x0 + text_w, W - 20)
            y1 = min(y0 + 72, H - 100)
            if x1 > x0 and y1 > y0:
                draw.rectangle([x0, y0, x1, y1], fill=color)
                draw.text((x0 + 8, y0 + 8), word, fill=CONFIG['COLOR_BG'])
            y_pos += 88
            if y_pos >= H - 120: break

        # Bottom strip
        draw.rectangle([0, H - 80, W, H], fill=(20, 29, 39))
        draw.text((24, H - 62),
                  f"{CONFIG['CHANNEL_NAME']}  --  Watch Now",
                  fill=CONFIG['COLOR_GOLD'])

        path = f'/content/thumbnails/thumb_{index}.jpg'
        img.save(path, 'JPEG', quality=95)
        print('  Thumbnail saved!')
        return path

    except Exception as e:
        print(f'  Thumbnail error: {e} -- using fallback')
        try:
            img  = Image.new('RGB', (W, H), CONFIG['COLOR_GREEN'])
            draw = ImageDraw.Draw(img)
            draw.text((40, H // 2 - 20), item['title'][:40], fill='white')
            path = f'/content/thumbnails/thumb_{index}.jpg'
            img.save(path, 'JPEG', quality=95)
            print('  Fallback thumbnail saved!')
            return path
        except:
            return None


print('Thumbnail generator loaded!')

In [ ]:
# CELL 10 - SEO GENERATOR AND GOOGLE DRIVE SAVER

def generate_seo(item):
    print('  Generating SEO...')
    type_ctx = {
        'pexels': 'motivational stock footage',
        'creative_commons': 'free tutorial',
        'podcast_clip': 'expert podcast clip'
    }.get(item['type'], 'curated video')

    raw = ask_groq(
        f'YouTube SEO for curated {type_ctx}.\n'
        f"Topic: \"{item['title']}\"\n"
        f"Niche: {item.get('keyword', 'make money online')}\n\n"
        f'Return ONLY valid JSON, no other text:\n'
        f'{{"title":"engaging title under 60 chars",'
        f'"description":"200 word SEO description with subscribe CTA",'
        f'"tags":["tag1","tag2","tag3","tag4","tag5","tag6","tag7","tag8","tag9","tag10"],'
        f'"thumbnail_text":"3 or 4 powerful words",'
        f'"category_id":"22"}}'
    )
    try:
        seo = json.loads(raw.replace('```json', '').replace('```', '').strip())
        print(f"  SEO ready: {seo['title']}")
        return seo
    except:
        print('  Using default SEO')
        return {
            'title': item['title'][:60],
            'description': 'Watch and subscribe for daily money making tips!',
            'tags': ['make money online', 'passive income', 'tutorial', 'YouTube tips'],
            'thumbnail_text': 'WATCH THIS NOW',
            'category_id': '22'
        }


def save_to_drive(video, thumb, seo, item, index):
    print('  Saving to Google Drive...')
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    date_str = datetime.now().strftime('%Y-%m-%d')
    folder   = f'/content/drive/MyDrive/AutoChannel/{date_str}/video_{index}'
    os.makedirs(folder, exist_ok=True)

    if video and os.path.exists(video): shutil.copy(video, f'{folder}/video.mp4')
    if thumb and os.path.exists(thumb): shutil.copy(thumb, f'{folder}/thumbnail.jpg')

    metadata = {
        'date': date_str, 'index': index,
        'content_type': item['type'], 'source_title': item['title'],
        'source_url': item.get('url', ''), 'seo_title': seo['title'],
        'description': seo['description'], 'tags': seo['tags'],
        'category_id': seo['category_id']
    }
    with open(f'{folder}/metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    print('  Saved to Drive!')
    return folder


print('SEO and Drive functions loaded!')

In [ ]:
# CELL 11 - AUTO YOUTUBE UPLOADER
# OAuth2 loaded from Colab Secrets vault

YT_SCOPES = [
    'https://www.googleapis.com/auth/youtube.upload',
    'https://www.googleapis.com/auth/youtube.force-ssl'
]


def setup_oauth_file():
    # OAuth2 hardcoded -- no secret key needed
    print('  Loading OAuth2 credentials...')
    # ── Load OAuth2 from Colab Secrets (never hardcode credentials!) ──
    creds_json = get_secret('OAUTH2_CREDENTIALS')
    if not creds_json:
        print()
        print('  ❌ OAUTH2_CREDENTIALS not found in Colab Secrets!')
        print('  Steps to fix:')
        print('  1. Go to console.cloud.google.com')
        print('  2. APIs & Services → Credentials → OAuth 2.0 Client IDs')
        print('  3. Download JSON → open in Notepad → copy ALL text')
        print('  4. Colab left sidebar → 🔑 icon → Add secret')
        print('  5. Name: OAUTH2_CREDENTIALS → paste JSON → toggle ON')
        return None
    path = '/content/client_secrets.json'
    with open(path, 'w') as f:
        f.write(creds_json)
    print('  OAuth2 loaded!')
    return path

def authenticate_youtube():
    print("Authenticating with YouTube...")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    token_path = "/content/drive/MyDrive/AutoChannel/yt_token.pickle"
    os.makedirs("/content/drive/MyDrive/AutoChannel", exist_ok=True)
    creds_path = setup_oauth_file()
    creds = None

    if os.path.exists(token_path):
        with open(token_path, "rb") as f:
            creds = pickle.load(f)
        print("  Found saved login token!")

    if creds and creds.expired and creds.refresh_token:
        print("  Refreshing token...")
        creds.refresh(Request())
        with open(token_path, "wb") as f:
            pickle.dump(creds, f)
        print("  Token refreshed!")

    elif not creds or not creds.valid:
        # Use device flow -- works on any phone, no redirect URI needed
        import urllib.parse, urllib.request, time as _time
        with open(creds_path) as f:
            client_data = json.load(f)["installed"]
        client_id     = client_data["client_id"]
        client_secret = client_data["client_secret"]
        scope = " ".join(YT_SCOPES)

        # Step 1: get device code
        r1 = urllib.request.urlopen(urllib.request.Request(
            "https://oauth2.googleapis.com/device/code",
            data=urllib.parse.urlencode({
                "client_id": client_id,
                "scope": scope
            }).encode(),
            headers={"Content-Type": "application/x-www-form-urlencoded"}
        ))
        device_data = json.loads(r1.read())

        print()
        print("=" * 60)
        print("  STEP 1: Open this URL in Chrome:")
        print()
        print(" ", device_data["verification_url"])
        print()
        print("  STEP 2: Enter this code on that page:")
        print()
        print("  ", device_data["user_code"])
        print()
        print("  STEP 3: Choose your YouTube Google account")
        print("  STEP 4: Tap Allow")
        print("  STEP 5: Come back here -- it will continue automatically!")
        print("=" * 60)
        print()

        # Step 2: poll until user approves
        device_code = device_data["device_code"]
        interval    = device_data.get("interval", 5)
        print("  Waiting for you to approve... (checks every 5 seconds)")
        while True:
            _time.sleep(interval)
            try:
                r2 = urllib.request.urlopen(urllib.request.Request(
                    "https://oauth2.googleapis.com/token",
                    data=urllib.parse.urlencode({
                        "client_id":     client_id,
                        "client_secret": client_secret,
                        "device_code":   device_code,
                        "grant_type":    "urn:ietf:params:oauth:grant-type:device_code"
                    }).encode(),
                    headers={"Content-Type": "application/x-www-form-urlencoded"}
                ))
                token_data = json.loads(r2.read())
                if "access_token" in token_data:
                    from google.oauth2.credentials import Credentials as _Creds
                    creds = _Creds(
                        token=token_data["access_token"],
                        refresh_token=token_data.get("refresh_token"),
                        token_uri="https://oauth2.googleapis.com/token",
                        client_id=client_id,
                        client_secret=client_secret,
                        scopes=YT_SCOPES
                    )
                    print("  Approved! Saving token...")
                    break
            except urllib.error.HTTPError as e:
                err = json.loads(e.read())
                if err.get("error") == "authorization_pending":
                    print("  Still waiting... approve on your phone")
                    continue
                elif err.get("error") == "slow_down":
                    interval += 5
                    continue
                else:
                    raise Exception(f"Auth error: {err}")

        with open(token_path, "wb") as f:
            pickle.dump(creds, f)
        print("  Login saved to Drive! Never needed again!")

    youtube = build("youtube", "v3", credentials=creds)
    print("  YouTube API connected!")
    return youtube

def upload_single_video(youtube, video_path, thumb_path, seo):
    print(f"  Uploading: {seo['title']}")
    body = {
        'snippet': {
            'title': seo['title'],
            'description': seo['description'],
            'tags': seo['tags'],
            'categoryId': seo.get('category_id', '22'),
            'defaultLanguage': 'en'
        },
        'status': {'privacyStatus': 'public', 'selfDeclaredMadeForKids': False}
    }
    media    = MediaFileUpload(video_path, mimetype='video/mp4',
                               resumable=True, chunksize=1024 * 1024 * 5)
    req      = youtube.videos().insert(part='snippet,status', body=body, media_body=media)
    response = None
    while response is None:
        status, response = req.next_chunk()
        if status:
            print(f'\r  Uploading... {int(status.progress() * 100)}%', end='', flush=True)
    print()

    video_id = response['id']
    yt_url   = f'https://youtube.com/watch?v={video_id}'
    print(f'  LIVE! --> {yt_url}')

    if thumb_path and os.path.exists(thumb_path):
        try:
            youtube.thumbnails().set(
                videoId=video_id,
                media_body=MediaFileUpload(thumb_path)
            ).execute()
            print('  Thumbnail set!')
        except Exception as e:
            print(f'  Thumbnail warning: {e}')

    playlist_id = CONFIG.get('PLAYLIST_ID', '').strip()
    if playlist_id:
        try:
            youtube.playlistItems().insert(
                part='snippet',
                body={'snippet': {
                    'playlistId': playlist_id,
                    'resourceId': {'kind': 'youtube#video', 'videoId': video_id}
                }}
            ).execute()
            print('  Added to playlist!')
        except Exception as e:
            print(f'  Playlist warning: {e}')

    return video_id, yt_url


def upload_all_todays_videos():
    print('Starting YouTube upload...')
    youtube  = authenticate_youtube()
    date_str = datetime.now().strftime('%Y-%m-%d')
    base     = f'/content/drive/MyDrive/AutoChannel/{date_str}'

    if not os.path.exists(base):
        print(f'No videos found for today ({date_str})')
        return []

    uploaded = []
    failed   = []

    for i in range(1, CONFIG['VIDEOS_PER_DAY'] + 1):
        folder     = f'{base}/video_{i}'
        meta_path  = f'{folder}/metadata.json'
        video_path = f'{folder}/video.mp4'
        thumb_path = f'{folder}/thumbnail.jpg'

        print(f'\n  Video {i} of {CONFIG["VIDEOS_PER_DAY"]}')
        print('  ' + '-' * 40)

        if not os.path.exists(folder) or not os.path.exists(video_path):
            print('  Files missing -- skipping')
            failed.append(i)
            continue

        with open(meta_path, 'r') as f:
            meta = json.load(f)

        if meta.get('youtube_video_id'):
            print(f"  Already uploaded: {meta.get('youtube_url', '')}")
            uploaded.append({'index': i, 'title': meta.get('seo_title', ''),
                             'url': meta.get('youtube_url', '')})
            continue

        try:
            seo = {
                'title': meta['seo_title'], 'description': meta['description'],
                'tags': meta['tags'], 'category_id': meta['category_id']
            }
            video_id, yt_url = upload_single_video(youtube, video_path, thumb_path, seo)
            meta.update({'youtube_video_id': video_id, 'youtube_url': yt_url,
                         'uploaded_at': datetime.now().isoformat()})
            with open(meta_path, 'w') as f:
                json.dump(meta, f, indent=2)
            uploaded.append({'index': i, 'title': seo['title'], 'url': yt_url})
            print(f'  Video {i} is LIVE!')
            if i < CONFIG['VIDEOS_PER_DAY']:
                print('  Waiting 30 seconds...')
                time.sleep(30)
        except Exception as e:
            print(f'  Upload failed: {e}')
            failed.append(i)

    print(f'\n  Uploaded: {len(uploaded)} | Failed: {len(failed)}')
    for v in uploaded:
        print(f"  --> {v['title']}")
        print(f"      {v['url']}")
    return uploaded


print('YouTube uploader loaded!')

In [ ]:
# CELL 12 - MASTER RUN CELL
# Runtime > Run All every day. That is all!

def run_full_daily_system():
    print()
    print('=' * 52)
    print('   AutoCuration Channel v6 -- Triple Voiceover')
    print('=' * 52)
    print(f"  Date:   {datetime.now().strftime('%A %B %d %Y')}")
    print(f"  Start:  {datetime.now().strftime('%H:%M')}")
    print(f"  Target: {CONFIG['VIDEOS_PER_DAY']} videos")
    print('  Voice:  ElevenLabs > Kokoro > Edge TTS')
    print('=' * 52)
    print()

    # Check required keys
    required = ['YOUTUBE_API_KEY', 'GROQ_API_KEY', 'PEXELS_API_KEY', 'CHANNEL_ID']
    missing  = [k for k in required if not CONFIG.get(k)]
    if missing:
        print('Cannot start -- missing Secrets:')
        for k in missing: print(f'  --> {k}')
        print('Tap key icon on left sidebar to add them')
        return

    all_results = []
    all_errors  = []

    # PHASE 1: FIND CONTENT
    print('PHASE 1: FINDING CONTENT')
    print('-' * 50)
    content_mix = get_todays_content_mix()
    if not content_mix:
        print('No content found -- check your API keys')
        return
    print()

    # PHASE 2: CREATE VIDEOS
    print('PHASE 2: CREATING VIDEOS')
    print('-' * 50)

    for i, item in enumerate(content_mix, 1):
        print()
        print(f'  VIDEO {i} of {len(content_mix)}')
        print(f"  Type:  {item['type'].upper()}")
        print(f"  Title: {item['title'][:55]}")
        print()
        try:
            src = download_content(item, i)
            if not src:
                print(f'  Download failed for video {i} -- skipping')
                all_errors.append({'index': i, 'reason': 'Download failed'})
                continue

            seo        = generate_seo(item)
            commentary = generate_commentary(item)
            audio      = generate_voiceover(commentary, f'/content/audio/vo_{i}.mp3')
            video      = process_video(src, audio, item, i)

            if not video:
                print(f'  Render failed for video {i} -- skipping')
                all_errors.append({'index': i, 'reason': 'Render failed'})
                continue

            thumb  = generate_thumbnail(item, seo, i)
            folder = save_to_drive(video, thumb, seo, item, i)
            all_results.append({'index': i, 'type': item['type'],
                                'title': seo['title'], 'folder': folder})
            print(f'  Video {i} complete!')

        except Exception as e:
            print(f'  Error on video {i}: {e}')
            all_errors.append({'index': i, 'reason': str(e)})
            continue

    print(f'\n  Phase 2 done -- {len(all_results)} videos created')

    if not all_results:
        print('  No videos created. Check errors above.')
        return

    # PHASE 3: UPLOAD
    print()
    print('PHASE 3: UPLOADING TO YOUTUBE')
    print('-' * 50)
    uploaded = upload_all_todays_videos()

    # FINAL SUMMARY
    print()
    print('=' * 52)
    print('           DAILY SYSTEM COMPLETE!')
    print('=' * 52)
    print(f'  Videos created:  {len(all_results)}')
    print(f'  Videos uploaded: {len(uploaded) if uploaded else 0}')
    print(f'  Errors:          {len(all_errors)}')
    print(f'  Finished:        {datetime.now().strftime("%H:%M")}')
    print('=' * 52)
    print()

    if uploaded:
        print('YOUR VIDEOS ARE LIVE ON YOUTUBE:')
        for v in uploaded:
            print(f"  --> {v['title']}")
            print(f"      {v['url']}")
        print()

    print('NOW PROMOTE ON SOCIAL MEDIA:')
    print('  WhatsApp Broadcast')
    print('  Facebook Groups (5 groups)')
    print('  Twitter / X')
    print('  Instagram Reels')
    print('  TikTok')
    print('  Telegram Channel')
    print()
    print('Come back tomorrow and run again!')
    return all_results, uploaded


# AUTO-RUNS WHEN YOU CLICK Runtime > Run All
run_full_daily_system()